# KATS — Experiment 1: Baseline Comparison (In-Distribution)

KATS Framework — Kinetic Attack Triage System


In [ ]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (f1_score, recall_score, precision_score,
                              cohen_kappa_score, classification_report)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Encode priority label: High=2, Medium=1, Low=0
label_map = {'Low': 0, 'Medium': 1, 'High': 2}

def prepare_xy(df, sample_n=None):
    d = df.copy()
    if sample_n:
        d = d.sample(n=min(sample_n, len(d)), random_state=42)
    d['priority_label_enc'] = d['priority_label'].map(label_map)
    # Encode service_sector
    d['sector_enc'] = LabelEncoder().fit_transform(d['service_sector'].astype(str))
    feature_cols = ['service_criticality','data_volume_gb','rto_minutes','rpo_minutes',
                    'dependency_count','downstream_critical','redundancy_level',
                    'regulatory_flag','active_sessions','bandwidth_required_mbps',
                    'latency_sensitivity','az_risk_score','multiregion_deployed',
                    'migration_complexity','sector_enc']
    X = d[feature_cols].fillna(0).astype(float)
    y = d['priority_label_enc']
    return X, y

# KATS-SYN is the primary training set
X_syn, y_syn = prepare_xy(df_kats_syn)
print(f"KATS-SYN  X: {X_syn.shape}  y dist: {y_syn.value_counts().to_dict()}")
print(f"Features: {X_syn.columns.tolist()}")

In [ ]:
# Rule-based baselines evaluated on full KATS-SYN (no CV needed — deterministic)
def rule_baseline_metrics(df, sort_col, ascending=False, name="B"):
    d = df.copy()
    d['rank'] = d[sort_col].rank(ascending=ascending, method='first')
    n = len(d)
    d['pred_label'] = np.where(d['rank'] <= n*0.30, 'High',
                       np.where(d['rank'] <= n*0.70, 'Medium', 'Low'))
    y_true = d['priority_label'].map(label_map)
    y_pred = d['pred_label'].map(label_map)
    return {
        'Baseline': name,
        'Macro_F1':     round(f1_score(y_true, y_pred, average='macro'), 4),
        'Recall_High':  round(recall_score(y_true, y_pred, labels=[2], average='macro'), 4),
        'Precision_High': round(precision_score(y_true, y_pred, labels=[2], average='macro', zero_division=0), 4),
        'Kappa':        round(cohen_kappa_score(y_true, y_pred), 4),
    }

def composite_rule_metrics(df, name="B3-Composite"):
    d = df.copy()
    d['comp_score'] = (0.5 * d['service_criticality']/10 +
                       0.3 * (1 - d['rto_minutes'].clip(0,1440)/1440) +
                       0.2 * d['az_risk_score'])
    d['rank'] = d['comp_score'].rank(ascending=False, method='first')
    n = len(d)
    d['pred_label'] = np.where(d['rank'] <= n*0.30, 'High',
                       np.where(d['rank'] <= n*0.70, 'Medium', 'Low'))
    y_true = d['priority_label'].map(label_map)
    y_pred = d['pred_label'].map(label_map)
    return {
        'Baseline': name,
        'Macro_F1':       round(f1_score(y_true, y_pred, average='macro'), 4),
        'Recall_High':    round(recall_score(y_true, y_pred, labels=[2], average='macro'), 4),
        'Precision_High': round(precision_score(y_true, y_pred, labels=[2], average='macro', zero_division=0), 4),
        'Kappa':          round(cohen_kappa_score(y_true, y_pred), 4),
    }

def deadline_first_heuristic(df, name="B6-DeadlineFirst"):
    d = df.copy()
    d['df_score'] = d['rto_minutes'] / (d['data_volume_gb'].clip(0.01))
    return rule_baseline_metrics(d, 'df_score', ascending=True, name=name)

def connectivity_rank(df, name="B7-ConnectivityRank"):
    d = df.copy()
    d['conn_score'] = d['dependency_count'] * d['service_criticality']
    return rule_baseline_metrics(d, 'conn_score', ascending=False, name=name)

rule_results = [
    rule_baseline_metrics(df_kats_syn, 'service_criticality', ascending=False, name="B1-Criticality"),
    rule_baseline_metrics(df_kats_syn, 'rto_minutes',         ascending=True,  name="B2-RTO"),
    composite_rule_metrics(df_kats_syn),
    deadline_first_heuristic(df_kats_syn),
    connectivity_rank(df_kats_syn),
]

# ML baselines via 5-fold stratified CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_metrics(estimator, X, y, name):
    scoring = ['f1_macro', 'recall_macro', 'precision_macro']
    cv_res = cross_validate(estimator, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    y_pred = np.zeros(len(y), dtype=int)
    for train_idx, test_idx in cv.split(X, y):
        estimator.fit(X.iloc[train_idx], y.iloc[train_idx])
        y_pred[test_idx] = estimator.predict(X.iloc[test_idx])
    return {
        'Baseline': name,
        'Macro_F1':       round(f1_score(y, y_pred, average='macro'), 4),
        'Recall_High':    round(recall_score(y, y_pred, labels=[2], average='macro'), 4),
        'Precision_High': round(precision_score(y, y_pred, labels=[2], average='macro', zero_division=0), 4),
        'Kappa':          round(cohen_kappa_score(y, y_pred), 4),
    }

b4 = cv_metrics(LogisticRegression(max_iter=500, random_state=42), X_syn, y_syn, "B4-LogReg")
b5 = cv_metrics(DecisionTreeClassifier(max_depth=10, random_state=42), X_syn, y_syn, "B5-DecTree")

all_baseline_results = rule_results + [b4, b5]
df_baselines = pd.DataFrame(all_baseline_results)
print("\n=== EXPERIMENT 1: BASELINE RESULTS ON KATS-SYN ===")
print(df_baselines.to_string(index=False))

In [ ]:
from sklearn.linear_model import LogisticRegression
import joblib, os

# Stage 1 base learners
rf   = RandomForestClassifier(n_estimators=200, max_depth=15,
                               class_weight={0:1,1:1,2:5},  # asymmetric loss via class_weight
                               random_state=42, n_jobs=-1)

lgbm = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05,
                            num_leaves=63, class_weight={0:1,1:1,2:5},
                            random_state=42, n_jobs=-1, verbose=-1)

nb   = CalibratedClassifierCV(GaussianNB(), method='isotonic', cv=3)

# Stage 2 meta-learner with asymmetric survivability loss (α=5)
meta = LogisticRegression(
    class_weight={0: 1, 1: 1, 2: 5},   # α=5 derived from threat model
    max_iter=1000, random_state=42, C=1.0
)

# KATS-Ensemble stacking
kats_ensemble = StackingClassifier(
    estimators=[('rf', rf), ('lgbm', lgbm), ('nb', nb)],
    final_estimator=meta,
    passthrough=True,           # pass original features to meta-learner too
    cv=5, n_jobs=-1
)

print("Training KATS-Ensemble on KATS-SYN (5-fold stacking)...")
kats_res = cv_metrics(kats_ensemble, X_syn, y_syn, "KATS-Ensemble")

# Also fit on full training set for later use
kats_ensemble.fit(X_syn, y_syn)
os.makedirs('/kaggle/working/models', exist_ok=True)
joblib.dump(kats_ensemble, '/kaggle/working/models/kats_ensemble.pkl')
joblib.dump(X_syn.columns.tolist(), '/kaggle/working/models/feature_cols.pkl')

print("\n=== KATS-Ensemble Result ===")
print(pd.DataFrame([kats_res]).to_string(index=False))

In [ ]:
df_all_results = pd.DataFrame(all_baseline_results + [kats_res])
df_all_results = df_all_results.sort_values('Recall_High', ascending=False).reset_index(drop=True)

print("\n" + "="*72)
print("EXPERIMENT 1 — FULL RESULTS (sorted by Recall_High)")
print("="*72)
print(df_all_results.to_string(index=False))

# Gap vs best baseline
best_baseline_recall = df_all_results[df_all_results['Baseline'] != 'KATS-Ensemble']['Recall_High'].max()
kats_recall          = df_all_results[df_all_results['Baseline'] == 'KATS-Ensemble']['Recall_High'].values[0]
best_baseline_f1     = df_all_results[df_all_results['Baseline'] != 'KATS-Ensemble']['Macro_F1'].max()
kats_f1              = df_all_results[df_all_results['Baseline'] == 'KATS-Ensemble']['Macro_F1'].values[0]

print(f"\n📊 KEY GAPS (KATS-Ensemble vs best baseline):")
print(f"   Recall_High : {kats_recall:.4f} vs {best_baseline_recall:.4f}  → +{kats_recall - best_baseline_recall:.4f} ({(kats_recall-best_baseline_recall)*100:.1f} pp)")
print(f"   Macro_F1    : {kats_f1:.4f} vs {best_baseline_f1:.4f}  → +{kats_f1 - best_baseline_f1:.4f} ({(kats_f1-best_baseline_f1)*100:.1f} pp)")

# Save results
df_all_results.to_csv('/kaggle/working/experiment1_results.csv', index=False)
print("\n✅ Saved to /kaggle/working/experiment1_results.csv")

In [ ]:
from scipy.stats import chi2

def mcnemar_test(y_true, y_pred_a, y_pred_b, name_a, name_b):
    """McNemars test: are the two models' errors significantly different?"""
    # Only look at cases where they disagree
    b = np.sum((y_pred_a == y_true) & (y_pred_b != y_true))  # A right, B wrong
    c = np.sum((y_pred_a != y_true) & (y_pred_b == y_true))  # A wrong, B right
    # With continuity correction
    if b + c == 0:
        return 1.0, 0.0
    chi2_stat = (abs(b - c) - 1)**2 / (b + c)
    p_value = 1 - chi2.cdf(chi2_stat, df=1)
    return round(p_value, 6), round(chi2_stat, 4)

# Re-run final fold predictions for KATS-Ensemble vs Decision Tree vs LogReg
from sklearn.model_selection import StratifiedKFold
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_for_test = {
    'KATS-Ensemble': kats_ensemble,
    'B5-DecTree':    DecisionTreeClassifier(max_depth=10, random_state=42),
    'B4-LogReg':     LogisticRegression(max_iter=2000, solver='saga', random_state=42),
    'B1-Criticality-rule': None,  # handled separately
}

preds_oof = {}
for name, model in [('KATS-Ensemble', kats_ensemble),
                     ('B5-DecTree',   DecisionTreeClassifier(max_depth=10, random_state=42)),
                     ('B4-LogReg',    LogisticRegression(max_iter=2000, solver='saga', random_state=42))]:
    oof = np.zeros(len(y_syn), dtype=int)
    for tr, te in cv5.split(X_syn, y_syn):
        model.fit(X_syn.iloc[tr], y_syn.iloc[tr])
        oof[te] = model.predict(X_syn.iloc[te])
    preds_oof[name] = oof

y_true_arr = y_syn.values

print("=== McNEMAR'S TEST: KATS-Ensemble vs Baselines ===\n")
print(f"{'Comparison':<40} {'b':>6} {'c':>6} {'chi2':>8} {'p-value':>10} {'Sig?':>6}")
print("-" * 78)
for bname in ['B5-DecTree', 'B4-LogReg']:
    p, chi2_s = mcnemar_test(y_true_arr, preds_oof['KATS-Ensemble'], preds_oof[bname],
                              'KATS-Ensemble', bname)
    b = np.sum((preds_oof['KATS-Ensemble'] == y_true_arr) & (preds_oof[bname] != y_true_arr))
    c = np.sum((preds_oof['KATS-Ensemble'] != y_true_arr) & (preds_oof[bname] == y_true_arr))
    sig = "✅ Yes" if p < 0.05 else "❌ No"
    print(f"{'KATS-Ensemble vs ' + bname:<40} {b:>6} {c:>6} {chi2_s:>8.3f} {p:>10.6f} {sig:>6}")

print("\n(b = cases KATS correct, baseline wrong; c = cases KATS wrong, baseline correct)")